# Vector Representation of Symptoms


Vector representations of symptoms were generated using the **intfloat/multilingual-e5-large** model, a transformer model from the E5 family optimized for semantic search and text similarity measurement.


## Package Installation

Installation of the necessary packages for working with the embedding model and the Neo4j graph database.


In [ ]:
%pip install sentence-transformers neo4j

In [ ]:
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase

## Loading the Model

Three models were evaluated during the selection process for the embedding model:
1. all-MiniLM-L6-v2
2. NeuML/pubmedbert-base-embeddings
3. intfloat/multilingual-e5-large

The evaluation results and the reasons for selecting a specific model are described in **embedding-test**.


In [ ]:
#model = SentenceTransformer('all-MiniLM-L6-v2')
#model = SentenceTransformer('NeuML/pubmedbert-base-embeddings')
model = SentenceTransformer('intfloat/multilingual-e5-large')

##  Connecting to the Neo4j Graph Database

Establishing a connection to the graph database.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

neo4j_url = os.getenv("NEO4J_URL")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(neo4j_url, auth=(neo4j_username, neo4j_password))

## Loading Symptoms from the Database

We load the symptoms from the graph database.


In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (s:Symptom)
        RETURN elementId(s) AS id, s.n4sch__label[0] AS label
    """)
    symptoms = [(record["id"], record["label"]) for record in result if record["label"]]

In [ ]:
symptoms

## Generating Vector Representations

We iterate through all symptoms from the graph database and generate their vector representations.


In [ ]:
labels = [label for _, label in symptoms]
embeddings = model.encode(labels)

In [ ]:
embeddings

## Writing Vector Representations to the Graph Database

The generated vector representations for each symptom are stored in the Neo4j graph database as attributes of the nodes representing the symptoms.


In [ ]:
with driver.session() as session:
    for (node_id, _), emb in zip(symptoms, embeddings):
        session.run("""
            MATCH (s)
            WHERE elementId(s) = $id
            SET s.embedding = $embedding
        """, id=node_id, embedding=emb.tolist())

## Verifying the Written Vector Representations

We verify that the vector representations have been successfully written.


In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (s:Symptom)
        RETURN elementId(s) AS id, s.embedding AS embedding
    """)
    for record in result:
        print(f"Node ID: {record['id']}, Embedding: {record['embedding']}")